 ## Cart2Insights — Step 5: Data Cleaning & Preprocessing

## Objective

Handle missing values

Remove duplicates

Correct data types

Format date/time columns

Standardize categorical values

Handle invalid records

Rename columns where required


Import Libraries

In [1]:
import pandas as pd
import numpy as np

Load datasets

In [2]:
customers = pd.read_csv('/content/drive/MyDrive/cart_to_insight/data/raw/olist_customers_dataset.csv')
geolocation = pd.read_csv('/content/drive/MyDrive/cart_to_insight/data/raw/olist_geolocation_dataset.csv')
order_items = pd.read_csv('/content/drive/MyDrive/cart_to_insight/data/raw/olist_order_items_dataset.csv')
order_payments = pd.read_csv('/content/drive/MyDrive/cart_to_insight/data/raw/olist_order_payments_dataset.csv')
order_reviews = pd.read_csv('/content/drive/MyDrive/cart_to_insight/data/raw/olist_order_reviews_dataset.csv')
orders = pd.read_csv('/content/drive/MyDrive/cart_to_insight/data/raw/olist_orders_dataset.csv')
products = pd.read_csv('/content/drive/MyDrive/cart_to_insight/data/raw/olist_products_dataset.csv')
sellers = pd.read_csv('/content/drive/MyDrive/cart_to_insight/data/raw/olist_sellers_dataset.csv')
category_translation = pd.read_csv('/content/drive/MyDrive/cart_to_insight/data/raw/product_category_name_translation.csv')

Dataset Inventory

In [3]:
datasets = {
    'customers': customers,
    'geolocation': geolocation,
    'order_items': order_items,
    'order_payments': order_payments,
    'order_reviews': order_reviews,
    'orders': orders,
    'products': products,
    'sellers': sellers,
    'category_translation': category_translation
}
for name, df in datasets.items():
    print(f"{name}: {df.shape}")

customers: (99441, 5)
geolocation: (1000163, 5)
order_items: (112650, 7)
order_payments: (103886, 5)
order_reviews: (99224, 7)
orders: (99441, 8)
products: (32951, 9)
sellers: (3095, 4)
category_translation: (71, 2)


### Inspecting Missing Values across all Dataframes

In [4]:
for name, df in datasets.items():
    missing_counts = df.isnull().sum()
    missing_percent = 100 * missing_counts / len(df)
    missing_df = pd.DataFrame({
        'Missing Count': missing_counts,
        'Percentage (%)': missing_percent
    }).query('`Missing Count` > 0')

    if not missing_df.empty:
        print(f"\n--- Missing Values in '{name}' ---")
        display(missing_df)
    else:
        print(f"'{name}' has no missing values.")

'customers' has no missing values.
'geolocation' has no missing values.
'order_items' has no missing values.
'order_payments' has no missing values.

--- Missing Values in 'order_reviews' ---


,Missing Count,Percentage (%)
review_comment_title,87656,88.341530
review_comment_message,58247,58.702532



--- Missing Values in 'orders' ---


,Missing Count,Percentage (%)
order_approved_at,160,0.160899
order_delivered_carrier_date,1783,1.793023
order_delivered_customer_date,2965,2.981668



--- Missing Values in 'products' ---


,Missing Count,Percentage (%)
product_category_name,610,1.851234
product_name_lenght,610,1.851234
product_description_lenght,610,1.851234
product_photos_qty,610,1.851234
product_weight_g,2,0.006070
product_length_cm,2,0.006070
product_height_cm,2,0.006070
product_width_cm,2,0.006070


'sellers' has no missing values.
'category_translation' has no missing values.


### Handling Missing Values

In [5]:
# 1. Handle missing values in order_reviews
order_reviews['review_comment_title'] = order_reviews['review_comment_title'].fillna('no_title')
order_reviews['review_comment_message'] = order_reviews['review_comment_message'].fillna('no_message')

# 2. Handle missing values in products
products['product_category_name'] = products['product_category_name'].fillna('unknown')

# Fill numerical product attributes with their respective medians
num_cols = [
    'product_name_lenght', 'product_description_lenght',
    'product_photos_qty', 'product_weight_g',
    'product_length_cm', 'product_height_cm', 'product_width_cm'
]
for col in num_cols:
    if col in products.columns:
        products[col] = products[col].fillna(products[col].median())

# 3. Handle missing values in orders (convert to datetime so missing values are standard NaT)
datetime_cols = [
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in datetime_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

print("Missing values successfully handled.")

Missing values successfully handled.


### Removing Duplicate Records

In [6]:
for name, df in datasets.items():
    duplicates_count = df.duplicated().sum()
    if duplicates_count > 0:
        print(f"Found {duplicates_count} duplicate rows in '{name}'. Removing them...")
        datasets[name] = df.drop_duplicates()
        # Also update global variables to reflect the change
        globals()[name] = datasets[name]
    else:
        print(f"No duplicate rows found in '{name}'.")

No duplicate rows found in 'customers'.
Found 261831 duplicate rows in 'geolocation'. Removing them...
No duplicate rows found in 'order_items'.
No duplicate rows found in 'order_payments'.
No duplicate rows found in 'order_reviews'.
No duplicate rows found in 'orders'.
No duplicate rows found in 'products'.
No duplicate rows found in 'sellers'.
No duplicate rows found in 'category_translation'.


### Correcting Data Types and Formatting Dates

In [7]:
# 1. Convert zip codes to string format to maintain consistency and keep leading zeros if any
customers['customer_zip_code_prefix'] = customers['customer_zip_code_prefix'].astype(str)
sellers['seller_zip_code_prefix'] = sellers['seller_zip_code_prefix'].astype(str)
geolocation['geolocation_zip_code_prefix'] = geolocation['geolocation_zip_code_prefix'].astype(str)

# 2. Format date/time columns across other datasets
order_items['shipping_limit_date'] = pd.to_datetime(order_items['shipping_limit_date'], errors='coerce')
order_reviews['review_creation_date'] = pd.to_datetime(order_reviews['review_creation_date'], errors='coerce')
order_reviews['review_answer_timestamp'] = pd.to_datetime(order_reviews['review_answer_timestamp'], errors='coerce')

print("Data types corrected and date/time columns formatted successfully!")

Data types corrected and date/time columns formatted successfully!


### Standardizing Categorical Values and Checking for Invalid Records

In [8]:
# 1. Standardize text features (convert to lower case and strip whitespace)
customers['customer_city'] = customers['customer_city'].str.strip().str.lower()
sellers['seller_city'] = sellers['seller_city'].str.strip().str.lower()
geolocation['geolocation_city'] = geolocation['geolocation_city'].str.strip().str.lower()

# 2. Check and correct invalid negative values in numerical columns
negative_prices = (order_items['price'] < 0).sum()
negative_freight = (order_items['freight_value'] < 0).sum()
negative_payments = (order_payments['payment_value'] < 0).sum()

print(f"Negative prices found: {negative_prices}")
print(f"Negative freight values found: {negative_freight}")
print(f"Negative payments found: {negative_payments}")

# If any negative values were found, we would correct them.
# Since they are expected to be zero or positive, let's enforce non-negative values:
order_items['price'] = order_items['price'].clip(lower=0)
order_items['freight_value'] = order_items['freight_value'].clip(lower=0)
order_payments['payment_value'] = order_payments['payment_value'].clip(lower=0)

print("Categorical values standardized and invalid numerical bounds handled.")

Negative prices found: 0
Negative freight values found: 0
Negative payments found: 0
Categorical values standardized and invalid numerical bounds handled.


### Renaming Columns Where Required

In [9]:
# Correcting the misspelling of 'length' in products dataframe
products = products.rename(columns={
    'product_name_lenght': 'product_name_length',
    'product_description_lenght': 'product_description_length'
})

# Update our dictionary reference as well
datasets['products'] = products

print("Columns renamed successfully! Updated product columns:")
print([col for col in products.columns if 'length' in col])

Columns renamed successfully! Updated product columns:
['product_name_length', 'product_description_length', 'product_length_cm']


### Export Cleaned Datasets to Google Drive

In [10]:
import os

output_dir = '/content/drive/MyDrive/cart_to_insight/data/cleaned'
os.makedirs(output_dir, exist_ok=True)

for name, df in datasets.items():
    file_path = os.path.join(output_dir, f"{name}_cleaned.csv")
    df.to_csv(file_path, index=False)
    print(f"Saved: {file_path}")

Saved: /content/drive/MyDrive/cart_to_insight/data/cleaned/customers_cleaned.csv
Saved: /content/drive/MyDrive/cart_to_insight/data/cleaned/geolocation_cleaned.csv
Saved: /content/drive/MyDrive/cart_to_insight/data/cleaned/order_items_cleaned.csv
Saved: /content/drive/MyDrive/cart_to_insight/data/cleaned/order_payments_cleaned.csv
Saved: /content/drive/MyDrive/cart_to_insight/data/cleaned/order_reviews_cleaned.csv
Saved: /content/drive/MyDrive/cart_to_insight/data/cleaned/orders_cleaned.csv
Saved: /content/drive/MyDrive/cart_to_insight/data/cleaned/products_cleaned.csv
Saved: /content/drive/MyDrive/cart_to_insight/data/cleaned/sellers_cleaned.csv
Saved: /content/drive/MyDrive/cart_to_insight/data/cleaned/category_translation_cleaned.csv
